In [ ]:
import re, json
from langchain_google_genai import ChatGoogleGenerativeAI
from llama_cpp import Llama
from langchain_community.chat_models import ChatLlamaCpp
from langchain_openai import ChatOpenAI
import os, ast
import datetime
from collections import defaultdict

current_date = datetime.date.today()

import asyncio
import string
from typing import List, Optional
from pydantic import BaseModel, Field
import numpy as np
# GPU settings (for ROCm)
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "10.3.0"
os.environ["HIP_VISIBLE_DEVICES"] = "0"
GEMINI_API_KEY = "API KEY"
OPENAI_API_KEY = "OPENAI API KEY"
OPENAI_BASE_URL = "https://openrouter.ai/api/v1"

In [ ]:

def load_model(model_name):
    model_dict = {
        'gemma_4b' : ['local',"/home/vijay/Downloads/models/gemma-3-gguf-gemma-3-4b-it-qat-q4_0-v3/gemma-3-4b-it-q4_0.gguf"],
        'mistral' : ['local',"/home/vijay/llama.cpp/models/mistral/mistral-7b-instruct-v0.2.Q4_K_M.gguf"],
        'llama_8b' : ['local',"/home/vijay/Downloads/models/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"],
        'gemma_1b' : ['local',"/home/vijay/Downloads/models/gemma-3-gguf-gemma-3-1b-it-qat-q4_0-v3/gemma-3-1b-it-q4_0.gguf"],
        'phi_3' : ['local',"/home/vijay/Downloads/models/phi-3-mini/Phi-3-mini-4k-instruct-q4.gguf"],
        'deepseek31' : ['openrouter','deepseek/deepseek-chat-v3.1:free'],
        'openai_oss' :['openrouter',"openai/gpt-oss-20b:free"],
        'mistral_24b' : ['openrouter',"mistralai/mistral-small-3.2-24b-instruct:free"],
        'deepseekr1' : ['openrouter',"deepseek/deepseek-r1-0528:free"],
        'llama_70b' : ['openrouter',"meta-llama/llama-3.3-70b-instruct:free"],
        'gemini-2.5_flash'   :['google', "gemini-2.5-flash-preview-09-2025"],
        'gemini-2.5_pro'   :['google',"gemini-2.5-pro"],
        'hermes_7b' :['local','/home/vijay/Downloads/models/Hermes-2-Pro-Llama-3-8B-Q4_K_M.gguf'],
        'gemini-2.0_flash':['google','gemini-2.0_flash'],
        'deepseek_r1': ['local', '/home/vijay/Downloads/models/DeepSeek-R1-0528-Qwen3-8B-Q4_K_M.gguf']
    }


    if model_dict[model_name][0] == 'local':
        extraction_model = ChatLlamaCpp(
            model_path = model_dict[model_name][1],
            n_ctx = 4096,
            n_gpu_layers = -1,
            verbose=False,
            max_tokens = 1024,
        )
    if model_dict[model_name][0] =='openrouter':
        extraction_model =  ChatOpenAI(
        model=model_dict[model_name][1],
        temperature=0.0,
        api_key=OPENAI_API_KEY,
        base_url=OPENAI_BASE_URL,
        )
    if model_dict[model_name][0] =='google':
        extraction_model = ChatGoogleGenerativeAI(
        model=model_dict[model_name][1],
        google_api_key=GEMINI_API_KEY,
        convert_system_message_to_human=True 
    )
    return extraction_model
tokenizer = Llama(  
    model_path = '/home/vijay/Downloads/models/all-MiniLM-L6-v2-Q8_0.gguf',
    verbose=False,
    n_gpu_layers=-1,
    n_ctx=512,
    embedding=True,
)
evaluator_llm = ChatGoogleGenerativeAI(
    model ="gemini-2.5-pro",
    google_api_key = GEMINI_API_KEY,
    convert_system_message_to_human = True
)



In [ ]:
def get_model_name(llm):
    for attr in ["model_name", "model", "model_path"]:
        if hasattr(llm, attr):
            return getattr(llm, attr)
    return None


# --- Constants ---
THROTTLE_WAIT_SECONDS = 1
THROTTLE_WAIT_GEMINI_PRO = 10
PUNCTUATION_TRANSLATOR = str.maketrans('', '', string.punctuation)

def normalize_string(text: str) -> str:
    """Removes punctuation, newlines, and extra whitespace for matching."""
    if not text:
        return ""
    # 1. Remove punctuation
    text_no_punct = text.translate(PUNCTUATION_TRANSLATOR)
    # 2. Lowercase and split into words (this handles newlines)
    words = text_no_punct.lower().split()
    # 3. Re-join with single spaces
    return " ".join(words)

# --- 1. Pydantic Models (No Change) ---
class Event(BaseModel):
    event_name: str = Field(description="The name of the event")
    people_involved: List[str] = Field(description="List of people involved in the event")
    date: str = Field(description="Date of the event in YYYY-MM-DD format. Use today's date if not specified.")
    location: Optional[str] = Field(description="Location of the event, leave empty if not specified", default="")
    source_sentence: str = Field(description="The exact sentence from the transcript that contains this information")

class Action(BaseModel):
    action_name: str = Field(description="The name of the action item")
    people_involved: List[str] = Field(description="List of people assigned to the action")
    date: str = Field(description="Due date of the action in YYYY-MM-DD format. Use today's date if not specified.")
    location: Optional[str] = Field(description="Location, leave empty if not specified", default="")
    source_sentence: str = Field(description="The exact sentence from the transcript that contains this information")

class TranscriptData(BaseModel):
    events: List[Event] = Field(description="A list of all extracted events")
    actions: List[Action] = Field(description="A list of all extracted action items")

class ParsedLine(BaseModel):
    line_id: int
    start_time: float
    end_time: float
    speaker: str
    text: str
    speaker_entity_id: int
    spokentime: datetime.datetime
    normalized_text: str 


async def extract_info_json(transcript: str) -> dict:
    chunks = chunk_text(transcript)
    dict_of_dicts = {}
    chunks_passed = []
    for i in range(len(chunks)):
        current_date = datetime.date.today().isoformat()
        prompt = f"""
        You are a useful information extraction system.
        Extract all events and action items from the transcript.
        The current date is: {current_date}. Use this if no other date is specified.
        
        Transcript:
        "{chunks[i]}"
        """

        try:
            # --- [THROTTLE] Wait before extraction (chat) call ---
            print(f"--- [Throttling] Waiting {THROTTLE_WAIT_SECONDS}s before extraction call... ---")
            await asyncio.sleep(THROTTLE_WAIT_SECONDS)
            
            structured_llm = extraction_model.with_structured_output(TranscriptData)
            response_data = await structured_llm.ainvoke(prompt)
            dict_of_dicts[i] = response_data.model_dump()
            chunks_passed.append(i)
        except Exception as e:
            print(f"Error during JSON extraction: {e}")
            
    return (dict_of_dicts,chunks,chunks_passed)

async def extract_info_json_llama(transcript: str) -> dict:
    chunks = chunk_text(transcript)
    dict_of_dicts = {}
    chunks_passed = []
    for i in range(len(chunks)):
        current_date = datetime.date.today().isoformat()
        prompt = f"""
        You are a useful information extraction system.
        Extract all events and action items from the transcript.
        The current date is: {current_date}. Use this if no other date is specified.

        Transcript:
        "{chunks[i]}"
        """

        try:
            
            structured_llm = extraction_model.with_structured_output(TranscriptData)
            response_data = await structured_llm.ainvoke(prompt)
            dict_of_dicts[i] = response_data.model_dump()
            chunks_passed.append(i)
        except Exception as e:
            print(f"Error during JSON extraction: {e}")
            
    return (dict_of_dicts,chunks,chunks_passed)


async def extract_info_chunks(chunks, current_model: list) -> dict:
    dict_of_dicts = {}
    chunks_passed = []
    for i in range(len(chunks)):
        current_date = datetime.date.today().isoformat()
        prompt = f"""
        You are a useful information extraction system.
        Extract all events and action items from the transcript.
        The current date is: {current_date}. Use this if no other date is specified.
        
        Transcript:
        "{chunks[i]}"
        """

        try:
            # --- [THROTTLE] Wait before extraction (chat) call ---
            print(f"--- [Throttling] Waiting {THROTTLE_WAIT_SECONDS}s before extraction call... ---")
            await asyncio.sleep(THROTTLE_WAIT_SECONDS)
            
            structured_llm = current_model.with_structured_output(TranscriptData)
            response_data = await structured_llm.ainvoke(prompt)
            dict_of_dicts[i] = response_data.model_dump()
            chunks_passed.append(i)
        except Exception as e:
            print(f"Error during JSON extraction: {e}")
            
    return (dict_of_dicts,chunks,chunks_passed)

class EvaluationResult(BaseModel):
    accuracy_score: int
    completeness_score: int
    format_score: int
    overall_score: int
    justification: str

def count_filled_fields(output_dict):
    dict_filled_fields= {
            'event_name': 0,
            'people_involved': 0,
            'date': 0,
            'location': 0,
            'source_sentence': 0, 
            'action_name': 0,
        }
    total_events = 0
    num_passed = 0
    num_failed =0 
    for i in output_dict.keys():
        try:
            TranscriptData(**output_dict[i])
            num_passed +=1
            sub_dict = output_dict[i]
            
            for item in sub_dict:
                events = sub_dict[item]
                if events:
                    num_events = len(sub_dict[item])
                    total_events += num_events
                    for event_dict in events:
                        for info in event_dict.keys():
                            if event_dict[info]:
                                dict_filled_fields[info] +=1
            
        except:
            num_failed +=1
            print('failed!')
    return [dict_filled_fields,total_events,num_passed,num_failed]

def calculate_averages(evals):
    score_dict, counts,num_chunks,chunks_passed = evals
    score_list_sum = [0,0,0,0]
    for i in score_dict.keys():
        for item in range(4):
            score_list_sum[item]+=score_dict[i][list(score_dict[i].keys())[item]]
    score_list_averages = [x/len(chunks_passed) for x in score_list_sum]
    pass_rate = len(chunks_passed)/num_chunks
    counts_list_average = [0,0,0,0]
    total_happenings = counts[1]
    for i in range(0,4):
        counts_list_average[i]+=counts[0][list(counts[0].keys())[i+1]]/total_happenings
    return(score_list_averages, pass_rate,counts_list_average)

def calculate_cosine(a,b):
    if type(a) == list:
        a_str= ','.join(map(str, a))
        b_str= ','.join(map(str, b))
    else:
        a_str = a
        b_str = b
    a_vec = np.array(tokenizer.embed(a_str))
    b_vec = np.array(tokenizer.embed(b_str))
    return np.dot(a_vec,b_vec)/(np.linalg.norm(a_vec)*np.linalg.norm(b_vec))

import numpy as np


def match_lists(golden_list, predicted_list, match_function, threshold):
    """
    Matches two lists of strings using a similarity function and threshold.
    
    This performs greedy matching to find the best pairs and correctly
    calculates TP, FP, and FN.
    """
    tp = 0
    
    # We need to keep track of which predicted items are "used"
    # to prevent one predicted item from matching two golden items.
    predicted_indices_used = set()
    
    # These are potential False Negatives
    unmatched_golden_items = []

    # Iterate through each item in the "ground truth" list
    for g_item in golden_list:
        best_score = -1
        best_p_index = -1

        # Find the best match for this golden item from the predicted list
        for i, p_item in enumerate(predicted_list):
            # Skip if this predicted item is already matched
            if i in predicted_indices_used:
                continue
                
            score = match_function(g_item, p_item)
            
            if score > best_score:
                best_score = score
                best_p_index = i

        # Check if the best match found is good enough (above threshold)
        if best_score >= threshold:
            tp += 1
            predicted_indices_used.add(best_p_index)
        else:
            # This golden item had no good match
            unmatched_golden_items.append(g_item)

    # False Negatives = golden items that were not matched
    fn = len(unmatched_golden_items)
    
    # False Positives = predicted items that were not used
    fp = len(predicted_list) - len(predicted_indices_used)

    return tp, fp, fn

def calculate_metrics(tp, fp, fn):
    """Calculates precision, recall, and F1-score."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return {'precision': precision, 'recall': recall, 'f1': f1}

# ---
# 3. MAIN EVALUATION ORCHESTRATOR
# ---

def evaluate_extraction_subdict(golden_dict, predicted_dict, threshold=0.8):
    """
    Evaluates the structured extraction against a golden dataset.
    """
    
    golden_events = golden_dict.get('events', [])
    predicted_events = predicted_dict.get('events', [])
    
    # Store detailed results
    results = {}
    
    # Store aggregate counts for the entire extraction
    total_tp = 0
    total_fp = 0
    total_fn = 0

    # ---
    # Step 1: Match the events list
    # ---
    
    # We need a "match function" for two event *objects*.
    # We'll use their 'event_name' as the key.
    def event_match_func(g_event, p_event):
        return calculate_cosine(g_event.get('event_name', ''), p_event.get('event_name', ''))

    # We need to find the *pairings* to evaluate sub-fields.
    # This is a more complex matching problem than the simple count.
    
    golden_indices_matched = set()
    predicted_indices_matched = set()
    
    event_matches = [] # Will store (golden_event, predicted_event) tuples

    # Greedy matching for event objects
    for g_idx, g_event in enumerate(golden_events):
        best_score = -1
        best_p_index = -1

        for p_idx, p_event in enumerate(predicted_events):
            if p_idx in predicted_indices_matched:
                continue
            
            score = event_match_func(g_event, p_event)
            if score > best_score:
                best_score = score
                best_p_index = p_idx
        
        if best_score >= threshold:
            event_matches.append((g_event, predicted_events[best_p_index]))
            golden_indices_matched.add(g_idx)
            predicted_indices_matched.add(best_p_index)

    # ---
    # Step 2: Calculate Event-level TP, FP, FN
    # ---
    event_tp = len(event_matches)
    event_fn = len(golden_events) - len(golden_indices_matched) # Unmatched golden events
    event_fp = len(predicted_events) - len(predicted_indices_matched) # Unmatched predicted events
    
    results['overall_events'] = {
        'tp': event_tp, 'fp': event_fp, 'fn': event_fn,
        **calculate_metrics(event_tp, event_fp, event_fn)
    }
    
    # Add to totals
    total_tp += event_tp
    total_fn += event_fn
    total_fp += event_fp

    # ---
    # Step 3: Evaluate sub-fields for matched events
    # ---
    
    for g_event, p_event in event_matches:
        event_name = g_event.get('event_name', 'Unknown Event')
        results[event_name] = {}
        
        # A) Evaluate 'people_involved' (List vs List)
        g_people = g_event.get('people_involved', [])
        p_people = p_event.get('people_involved', [])
        
        p_tp, p_fp, p_fn = match_lists(g_people, p_people, calculate_cosine, threshold)
        results[event_name]['people_involved'] = {
            'tp': p_tp, 'fp': p_fp, 'fn': p_fn,
            **calculate_metrics(p_tp, p_fp, p_fn)
        }
        total_tp += p_tp
        total_fp += p_fp
        total_fn += p_fn
        
        # B) Evaluate 'location' (String vs String)
        g_people_raw = g_event.get('people_involved', [])
        p_people_raw = p_event.get('people_involved', [])

        # Filter out any None values from within the lists
        g_loc = [person for person in g_people_raw if person is not None]
        p_loc = [person for person in p_people_raw if person is not None]
        loc_score = calculate_cosine(g_loc, p_loc)
        l_tp, l_fp, l_fn = 0, 0, 0
        
        if loc_score >= threshold:
            l_tp = 1 # Matched
        else:
            if g_loc: # Golden had a value
                l_fn = 1 # Model missed it
            if p_loc: # Model predicted a value
                l_fp = 1 # Model hallucinated it
        
        # Handle the case where both are correctly empty
        if not g_loc and not p_loc:
             l_tp, l_fp, l_fn = 1, 0, 0
             
        results[event_name]['location'] = {
            'tp': l_tp, 'fp': l_fp, 'fn': l_fn,
            **calculate_metrics(l_tp, l_fp, l_fn)
        }
        total_tp += l_tp
        total_fp += l_fp
        total_fn += l_fn

    # ---
    # Step 4: Final Report
    # -
    return results

# ---
async def evaluate_quality(result):
    output_dict, chunks, chunks_passed = result
    evals = {}
    for i in chunks_passed:
        try:
            # --- [THROTTLE] Wait before evaluating ---
            print(f"--- [Throttling] Waiting {THROTTLE_WAIT_GEMINI_PRO}s before evaluating... ---")
            await asyncio.sleep(THROTTLE_WAIT_GEMINI_PRO) 
            current_date = datetime.date.today().isoformat()
            prompt = f"""
            You are a useful evaluator system. Your job is to evaluate the quality of extracted structured data from transcripts

            Below is the transcript, followed by the extracted dictionary. Current date is {current_date}
            Score it on a scale of 1-10, and keep your remarks concise.

            ---
            Transcript:
            \"\"\"{chunks[i]}\"\"\"

            Extracted Dictionary:
            {json.dumps(output_dict[i], indent=2)}
            """
            structured_eval = evaluator_llm.with_structured_output(EvaluationResult)
            response = await structured_eval.ainvoke(prompt)
            evaluation = response.model_dump()
            evals[i]= evaluation
        except:
            print("evaluation failed!")
    return evals

async def overall_evaluation(result):
    gemini_score = await evaluate_quality(result)
    output_dict, chunks, chunks_passed = result
    basic_metrics = count_filled_fields(output_dict)
    return [gemini_score, basic_metrics,len(chunks),chunks_passed]

def evaluate_extraction(pred_dict,gold_dict):
    results = {}
    results['overall_events'] = {
        'tp': 0, 'fp': 0, 'fn': 0, 'precision': 0, 'recall': 0, 'f1': 0,
    }
    for i in pred_dict.keys():
        result_i = evaluate_extraction_subdict(pred_dict[i],gold_dict[i])
        for j in result_i['overall_events'].keys():
            results['overall_events'][j]+=result_i['overall_events'][j]

    print("--- LLM Extraction Evaluation Report ---")
    print("\n")
    
    print("### Overall Event-Level Metrics ###")
    report = results['overall_events']
    print(f"Event F1: {report['f1']:.2%}")
    print(f"  - True Positives:  {report['tp']} (Events matched)")
    print(f"  - False Positives: {report['fp']} (Model hallucinated events)")
    print(f"  - False Negatives: {report['fn']} (Model missed events)")
    print("\n---\n")
    
    print("### Detailed Field-Level Metrics (for Matched Events) ###")
    for event_name, fields in results.items():
        if event_name == 'overall_events':
            continue
        print(f"\nEvent: '{event_name}'")
        for field, metrics in fields.items():
            print(f"  - Field: '{field}'")
            print(f"    - F1: {metrics['f1']:.2%}")
            print(f"    - (TP: {metrics['tp']}, FP: {metrics['fp']}, FN: {metrics['fn']})")
    
    print("\n---\n")
    print("### Total Micro-Averaged Metrics (all fields) ###")
    total_tp = results['overall_events']['tp']
    total_fp = results['overall_events']['fp']
    total_fn = results['overall_events']['fn']
    total_metrics = calculate_metrics(total_tp, total_fp, total_fn)
    print(f"**Overall F1-Score: {total_metrics['f1']:.2%}**")
    print(f"  - Total True Positives:  {total_tp}")
    print(f"  - Total False Positives: {total_fp}")
    print(f"  - Total False Negatives: {total_fn}")
    print(f"  - Overall Precision:   {total_metrics['precision']:.2%}")
    print(f"  - Overall Recall:      {total_metrics['recall']:.2%}")
    
    return results

async def evaluate_golden(chunks,golden,model):
    result = await extract_info_chunks(chunks,model)
    result_fields = await overall_evaluation(result)
    try:
        averaged_score = calculate_averages(result_fields)
    except:
        averaged_score = []
    try:
        result_similarity = evaluate_extraction(result[0],golden)
    except:
        result_similarity =[]
    sentiment  = await evaluate_sentiments(result_fields)
    return (averaged_score,result_fields, result_similarity,sentiment)

async def evaluate_sentiments(result_fields):
    gemini_score = result_fields[0]
    critique_list = []
    
    for i in gemini_score.keys():
        justification = gemini_score[i]['justification']
        critique_list.append(justification)
    critique_string = '\n'.join(critique_list)
    prompt = f'This is the output of an evaluator LLM which has evaluated the output of another LLM. Summarise this {critique_string}'
    response = await evaluator_llm.ainvoke(prompt)
    output =response.model_dump()['content']
    print(output)


In [ ]:

model_dict = {
    'gemma_4b' : ['local',"/home/vijay/Downloads/models/gemma-3-gguf-gemma-3-4b-it-qat-q4_0-v3/gemma-3-4b-it-q4_0.gguf"],
    'mistral' : ['local',"/home/vijay/llama.cpp/models/mistral/mistral-7b-instruct-v0.2.Q4_K_M.gguf"],
    'llama_8b' : ['local',"/home/vijay/Downloads/models/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"],
    'gemma_1b' : ['local',"/home/vijay/Downloads/models/gemma-3-gguf-gemma-3-1b-it-qat-q4_0-v3/gemma-3-1b-it-q4_0.gguf"],
    'phi_3' : ['local',"/home/vijay/Downloads/models/phi-3-mini/Phi-3-mini-4k-instruct-q4.gguf"],
    'deepseek31' : ['openrouter','deepseek/deepseek-chat-v3.1:free'],
    'openai_oss' :['openrouter',"openai/gpt-oss-20b:free"],
    'mistral_24b' : ['openrouter',"mistralai/mistral-small-3.2-24b-instruct:free"],
    'deepseekr1' : ['openrouter',"deepseek/deepseek-r1-0528:free"],
    'llama_70b' : ['openrouter',"meta-llama/llama-3.3-70b-instruct:free"],
    'gemini-2.5_flash'   :['google', "gemini-2.5-flash-preview-09-2025"],
    'gemini-2.5_pro'   :['google',"gemini-2.5-pro"],
    'hermes_7b' :['local','/home/vijay/Downloads/models/Hermes-2-Pro-Llama-3-8B-Q4_K_M.gguf'],
    'gemini-2.0_flash':['google','gemini-2.0_flash'],
    'deepseek_r1': ['local', '/home/vijay/Downloads/models/DeepSeek-R1-0528-Qwen3-8B-Q4_K_M.gguf']
}


In [ ]:
def chunk_transcript(transcript: str, model_max_tokens: int = 4096, reserved_tokens: int = 512):
    tokens = extraction_model_llama.client.tokenize(transcript.encode("utf-8"))
    chunk_size = model_max_tokens - reserved_tokens
    chunks = []
    for i in range(0, len(tokens), chunk_size):
        chunk_tokens = tokens[i:i + chunk_size]
        chunk_text = extraction_model_llama.client.detokenize(chunk_tokens).decode("utf-8",errors="ignore")
        chunks.append(chunk_text)
    return chunks

################################################################################

def chunk_text(text, chunk_size=1000, overlap=200):
    """Split text into overlapping chunks by tokens."""
    tokens = extraction_model.client.tokenize(text.encode('utf-8'))
    chunks = []
    for i in range(0, len(tokens), chunk_size - overlap):
        chunk = tokens[i:i + chunk_size]
        chunks.append(extraction_model.client.detokenize(chunk).decode('utf-8', errors = "ignore"))
    return chunks
async def evaluator(file_path, model):
    extraction_model = load_model(model)

    with open(file_path,'r') as f:
        transcript = f.read()

    event_chunks = re.split(r'\n(?=\d+\.\s)', transcript.strip())

    # This list will hold our final parsed data
    parsed_data = []

    for chunk in event_chunks:
        try:
            # Find the start of the JSON block
            # We use rfind to get the LAST '{' which is the start of our JSON object
            json_start_index = chunk.find('{')
            
            if json_start_index == -1:
                # No JSON found in this chunk, skip
                print(f"Skipping chunk, no JSON object found:\n{chunk[:100]}...")
                continue
                
            # Extract the text part (everything before the JSON)
            text_part = chunk[:json_start_index].strip()
            
            # Clean up the "JSON" marker from the examples
            if text_part.endswith('JSON'):
                text_part = text_part[:-4].strip()
                
            # Extract the JSON string
            json_string = chunk[json_start_index:].strip()
            
            # The first example has "JSON" *after* the brace, remove it
            if json_string.endswith('JSON'):
                json_string = json_string[:-4].strip()

            # Parse the JSON string into a Python dictionary
            data_dict = json.loads(json_string)
            
            # Store the results as a dictionary
            parsed_data.append({
                'text_content': text_part,
                'json_data': data_dict
            })
            
        except json.JSONDecodeError as e:
            print(f"Error parsing JSON for a chunk: {e}")
            print(f"Problematic JSON string: {json_string[:200]}...")
        except Exception as e:
            print(f"An unexpected error occurred with a chunk: {e}")

    # --- You can now work with the 'parsed_data' list ---

    print(f"\nSuccessfully parsed {len(parsed_data)} events.")


    # To see the full first item's dictionary:
    # import pprint
    # pprint.pprint(parsed_data[0]['json_data'])

    chunks = [parsed_data[i]['text_content'] for i in range(len(parsed_data))]
    golden = [parsed_data[i]['json_data'] for i in range(len(parsed_data))]
    golden_cut = golden[0:10]
    chunk_cut = chunks[0:10]
    result =  await evaluate_golden(chunks=chunk_cut,golden=golden_cut,model=extraction_model)
    filename = f'{model}.txt'
    with open(filename ,'w') as f:
        f.write(json.dumps(result, indent=2, ensure_ascii=False))
    



    if hasattr(extraction_model, "client") and hasattr(extraction_model.client, "close"):
        extraction_model.client.close()


In [ ]:
model = 'gemini-2.5_flash'
print(f'running {model}')
await evaluator('evals/golden_dataset.txt',model)
